<a href="https://colab.research.google.com/github/vivianlinnn/DS41_IDXExchange/blob/main/src/07_LightGBM_Team.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# START : Anjali Manju Gowda

# Install LightGBM
This cell installs the LightGBM library, which is a gradient boosting framework optimized for speed and performance.

In [1]:
!pip install lightgbm

# Import Required Libraries
We import core Python libraries for data handling, numerical computation, and model evaluation:
- `pandas` & `numpy` → data manipulation and numerical operations
- `LGBMRegressor` → LightGBM gradient boosting regression model
- `r2_score` & `mean_absolute_percentage_error` → metrics to evaluate model performance

In [2]:
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score, mean_absolute_percentage_error

# Load Training and Test Data
We load the cleaned feature-engineered CSV datasets:
- `train_cleaned_fe.csv` → training set
- `test_cleaned_fe.csv` → testing set
The `on_bad_lines='skip'` ensures malformed rows are ignored.

In [3]:
# Load CSV files
train = pd.read_csv('../processed_data/train_cleaned_fe.csv', engine='python', on_bad_lines='skip')
test  = pd.read_csv('../processed_data/test_cleaned_fe.csv', engine='python', on_bad_lines='skip')

# Convert Columns to Numeric
Ensures that numeric columns are properly typed to prevent errors during modeling. Any non-numeric or invalid entries are converted to NaN.

In [4]:
# Convert numeric columns safely
numeric_cols = [
    'LivingArea','BedroomsTotal','LotSizeSquareFeet',
    'BathroomsTotalInteger','Bed_Bath_Ratio',
    'Property_Age','Months_From_Dec_2025'
]
for col in numeric_cols:
    train[col] = pd.to_numeric(train[col], errors='coerce')
    test[col]  = pd.to_numeric(test[col], errors='coerce')

train['ClosePrice'] = pd.to_numeric(train['ClosePrice'], errors='coerce')
test['ClosePrice']  = pd.to_numeric(test['ClosePrice'], errors='coerce')

# Remove Rows with Missing Target
Rows where the target variable `ClosePrice` is missing are dropped to ensure the model only trains on valid data.

In [5]:
# Drop rows with missing target
train = train.dropna(subset=['ClosePrice'])
test  = test.dropna(subset=['ClosePrice'])

# Log-transform the Target Variable
The `ClosePrice` is right-skewed. Taking `log(ClosePrice)`:
- Stabilizes variance
- Reduces influence of extremely high-priced properties
- Improves model performance

In [6]:
# Log-transform the target
train['LogClosePrice'] = np.log(train['ClosePrice'])
test['LogClosePrice']  = np.log(test['ClosePrice'])

# Create Additional Features
We generate new features to improve predictive power:
- Replace zeros with NaN for Bedrooms and LotSize to prevent division errors
- `Sqft_Per_Bedroom` → living area per bedroom
- `Lot_Utilization` → efficiency of land usage (living area / lot size)

In [7]:
# Feature engineering (example)
for df in [train, test]:
    df['BedroomsTotal'] = df['BedroomsTotal'].replace(0, np.nan)
    df['LotSizeSquareFeet'] = df['LotSizeSquareFeet'].replace(0, np.nan)
    df['Sqft_Per_Bedroom'] = df['LivingArea'] / df['BedroomsTotal']
    df['Lot_Utilization']  = df['LivingArea'] / df['LotSizeSquareFeet']

# Encode Location Using Median Price per ZIP
Instead of one-hot encoding ZIP codes, we use the median log price for each ZIP:
- Captures neighborhood-level pricing effects
- Reduces high dimensionality
- Unseen ZIPs in test set are assigned the global median

In [8]:
# ZIP-level median price encoding
zip_median = train.groupby('PostalCode')['LogClosePrice'].median()
train['ZipMedianPrice'] = train['PostalCode'].map(zip_median)
test['ZipMedianPrice']  = test['PostalCode'].map(zip_median)
global_median = train['LogClosePrice'].median()
test['ZipMedianPrice'] = test['ZipMedianPrice'].fillna(global_median)

# Define Model Features
The `tree_features` list includes:
- Location (`ZipMedianPrice`)
- Property attributes (`LivingArea`, `BathroomsTotalInteger`, `BedroomsTotal`)
- Engineered features (`Bed_Bath_Ratio`, `Sqft_Per_Bedroom`, `Lot_Utilization`)
- Time/age features (`Property_Age`, `Months_From_Dec_2025`)

In [9]:
# Select features
tree_features = [
    'ZipMedianPrice','LivingArea','BathroomsTotalInteger',
    'BedroomsTotal','Bed_Bath_Ratio','Property_Age',
    'Months_From_Dec_2025','Sqft_Per_Bedroom','Lot_Utilization'
]

# Handle Missing and Infinite Values
Ensures all feature columns contain valid numerical values:
- Replace `inf` and `-inf` with NaN
- Drop rows with missing values in features or target

In [10]:
# Drop any remaining missing values
for df in [train, test]:
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(subset=tree_features + ['LogClosePrice'], inplace=True)


# Prepare Feature Matrices and Target Vectors
Split data into:
- `X_train` / `X_test` → feature matrices
- `y_train` / `y_test` → target variable (`LogClosePrice`)

In [11]:
# Prepare train/test matrices
X_train = train[tree_features]
y_train = train['LogClosePrice']
X_test  = test[tree_features]
y_test  = test['LogClosePrice']

# Train LightGBM Regressor
- Gradient boosting model (`LGBMRegressor`)
- Parameters:
  - `n_estimators=200` → number of trees
  - `max_depth=6` → max depth per tree to avoid overfitting
  - `learning_rate=0.1` → step size for each boosting iteration
- Trains a sequence of decision trees to minimize prediction error

In [12]:
# Train LightGBM
lgbm_model = LGBMRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42
)
lgbm_model.fit(X_train, y_train)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001005 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1236
[LightGBM] [Info] Number of data points in the train set: 67669, number of used features: 9
[LightGBM] [Info] Start training from score 13.773877
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain:

LGBMRegressor(max_depth=6, n_estimators=200, random_state=42)

# Generate Predictions
Predict log-transformed prices for:
- Training set (`train_pred`)
- Testing set (`test_pred`)

In [13]:
# Predictions
train_pred = lgbm_model.predict(X_train)
test_pred  = lgbm_model.predict(X_test)


# Convert Predictions Back to Dollar Scale
Exponentiate log predictions to return to actual price scale for evaluation:
- `y_train_d`, `y_test_d` → original target
- `train_pred_d`, `test_pred_d` → predicted prices

In [14]:
# Convert back to original scale
y_train_d = np.exp(y_train)
y_test_d  = np.exp(y_test)
train_pred_d = np.exp(train_pred)
test_pred_d  = np.exp(test_pred)

# Evaluate Model Performance
Metrics used:
- `R²` → proportion of variance explained
- `MAPE` → mean absolute percentage error
- `MdAPE` → median absolute percentage error (robust to outliers)
This gives an understanding of both accuracy and reliability of predictions.

In [15]:
# Metrics
train_r2 = r2_score(y_train, train_pred)
test_r2  = r2_score(y_test, test_pred)
train_mape = mean_absolute_percentage_error(y_train_d, train_pred_d) * 100
test_mape  = mean_absolute_percentage_error(y_test_d, test_pred_d) * 100
train_mdape = np.median(np.abs((y_train_d - train_pred_d) / y_train_d)) * 100
test_mdape  = np.median(np.abs((y_test_d - test_pred_d) / y_test_d)) * 100

print("LightGBM Results")
print(f"Train R²: {train_r2:.4f}, Test R²: {test_r2:.4f}")
print(f"Train MAPE: {train_mape:.2f}%, Test MAPE: {test_mape:.2f}%")
print(f"Train MdAPE: {train_mdape:.2f}%, Test MdAPE: {test_mdape:.2f}%")

LightGBM Results
Train R²: 0.9159, Test R²: 0.8903
Train MAPE: 13.24%, Test MAPE: 14.84%
Train MdAPE: 9.69%, Test MdAPE: 10.50%


# LightGBM Model Performance Summary

The LightGBM regression model was trained on log-transformed property prices using feature-engineered variables, including ZIP-level median prices, property attributes, and derived metrics such as `Sqft_Per_Bedroom` and `Lot_Utilization`.

## Key Evaluation Metrics

| Metric       | Training Set | Testing Set |
|-------------|-------------|------------|
| **R²**      | 0.916       | 0.890      |
| **MAPE (%)**| 13.24       | 14.84      |
| **MdAPE (%)**| 9.69       | 10.50      |

### Interpretation

1. **High R²**: The model explains ~92% of variance on training data and ~89% on unseen test data, indicating strong predictive power and reasonable generalization.  
2. **Low MAPE and MdAPE**: The mean and median absolute percentage errors are low, meaning predictions are, on average, within ~15% of the true property prices, with typical errors closer to 10%.  
3. **Robustness**: The small difference between training and testing metrics suggests the model is not overfitting and handles unseen data well.  
4. **Practical Use**: This LightGBM model effectively captures non-linear relationships between property features and prices, making it suitable for real estate price estimation and decision-making.

> Overall, the LightGBM model provides an accurate, interpretable, and robust approach for predicting residential property closing prices using feature-engineered data.

# START: LEXI CHEN

# Hyperparameter Tuning for LightGBM

The baseline LightGBM model above uses a fixed set of parameters. That gives us a strong starting point, but it does not tell us whether the model is too simple, too complex, or leaving performance on the table.

In this section, we tune the model to find a better balance between **bias, variance, and generalization**.

### Why tune these hyperparameters?

- **`num_leaves` and `max_depth`** control tree complexity.  
  Larger values can capture more nonlinear patterns, but they can also overfit.

- **`min_child_samples`** controls how many observations are required in a leaf.  
  Higher values make the model more conservative and often improve generalization.

- **`learning_rate`** controls how quickly the model learns.  
  Smaller values usually need more trees, but they can produce more stable results.

- **`n_estimators`** controls the number of boosting rounds.  
  More trees can improve fit, but only if paired with an appropriate learning rate.

- **`subsample` and `colsample_bytree`** introduce randomness.  
  This can reduce overfitting by preventing every tree from seeing all rows and all features.

- **`reg_alpha` and `reg_lambda`** add L1/L2 regularization.  
  These help control overly aggressive splits and improve robustness.

We use **cross-validation** so that the selected hyperparameters are judged on multiple validation folds instead of one lucky split. This makes the tuning process more reliable.


In [17]:
from sklearn.model_selection import RandomizedSearchCV, KFold
from sklearn.metrics import mean_squared_error

# Cross-validation strategy
cv_strategy = KFold(n_splits=5, shuffle=True, random_state=42)

# Hyperparameter search space
param_distributions = {
    'n_estimators': [100, 200, 300, 500, 700],
    'learning_rate': [0.01, 0.03, 0.05, 0.08, 0.1],
    'max_depth': [3, 4, 5, 6, 8, 10, -1],
    'num_leaves': [15, 31, 50, 70, 100, 150],
    'min_child_samples': [5, 10, 20, 30, 50],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'reg_alpha': [0, 0.01, 0.1, 1, 5],
    'reg_lambda': [0, 0.01, 0.1, 1, 5]
}

# Base estimator for search
lgbm_search_model = LGBMRegressor(
    objective='regression',
    random_state=42
)

# Randomized search is more efficient than a full grid search because
# it explores many combinations without trying every possible one.
random_search = RandomizedSearchCV(
    estimator=lgbm_search_model,
    param_distributions=param_distributions,
    n_iter=8,
    scoring='neg_root_mean_squared_error',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

print("Best cross-validation RMSE (log scale):", -random_search.best_score_)
print("Best hyperparameters:")
print(random_search.best_params_)


Fitting 3 folds for each of 8 candidates, totalling 24 fits
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.011935 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1226
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.012399 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1229
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.011273 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1229
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008151 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1226
[LightGBM] [Info] Number of data points in the train set: 45113, number of used features: 9


## Train and Evaluate the Tuned Model

Now we take the best hyperparameter combination found during cross-validation and train a new LightGBM model on the full training data.

This step matters because:

1. **Cross-validation chooses the most reliable parameter set** based on repeated validation performance.
2. **Retraining on all available training data** gives the tuned model the most information possible.
3. We can then compare the tuned model against the original baseline on both the training and test sets to see whether tuning improved generalization rather than just memorizing the training data.


In [18]:
# Train tuned model using the best parameters from the search
best_lgbm_model = random_search.best_estimator_
best_lgbm_model.fit(X_train, y_train)

# Predictions from the tuned model
tuned_train_pred = best_lgbm_model.predict(X_train)
tuned_test_pred = best_lgbm_model.predict(X_test)

# Convert predictions back to dollar scale
tuned_train_pred_d = np.exp(tuned_train_pred)
tuned_test_pred_d = np.exp(tuned_test_pred)

# Evaluate tuned model
tuned_train_r2 = r2_score(y_train, tuned_train_pred)
tuned_test_r2 = r2_score(y_test, tuned_test_pred)

tuned_train_mape = mean_absolute_percentage_error(y_train_d, tuned_train_pred_d) * 100
tuned_test_mape = mean_absolute_percentage_error(y_test_d, tuned_test_pred_d) * 100

tuned_train_mdape = np.median(np.abs((y_train_d - tuned_train_pred_d) / y_train_d)) * 100
tuned_test_mdape = np.median(np.abs((y_test_d - tuned_test_pred_d) / y_test_d)) * 100

print("Tuned LightGBM Results")
print(f"Train R²: {tuned_train_r2:.4f}, Test R²: {tuned_test_r2:.4f}")
print(f"Train MAPE: {tuned_train_mape:.2f}%, Test MAPE: {tuned_test_mape:.2f}%")
print(f"Train MdAPE: {tuned_train_mdape:.2f}%, Test MdAPE: {tuned_test_mdape:.2f}%")


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000590 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1236
[LightGBM] [Info] Number of data points in the train set: 67669, number of used features: 9
[LightGBM] [Info] Start training from score 13.773877
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Tuned LightGBM Results
Train R²: 0.9205, Test R²: 0.8925
Train MAPE: 12.75%, Test MAPE: 14.58%
Train MdAPE: 9.21%, Test MdAPE: 10.12%


## Interpretation of Hyperparameter Tuning Results

Hyperparameter tuning was performed using **RandomizedSearchCV with 3-fold cross-validation** to efficiently search for improved LightGBM parameters while keeping computational cost manageable. Instead of exhaustively evaluating all possible combinations as in GridSearchCV, randomized search samples combinations from the parameter space, allowing us to explore important parameters while reducing training time.

The best hyperparameters identified were:

- learning_rate = 0.05  
- n_estimators = 200  
- num_leaves = 150  
- max_depth = -1  
- min_child_samples = 10  
- subsample = 0.9  
- colsample_bytree = 0.8  
- reg_alpha = 5  
- reg_lambda = 1  

These parameters balance **model complexity and generalization**. A moderate learning rate combined with 200 boosting iterations allows the model to learn gradually and capture patterns in the data without overfitting. Subsampling parameters (`subsample` and `colsample_bytree`) introduce randomness into the training process, which improves model robustness. Additionally, the regularization parameters (`reg_alpha` and `reg_lambda`) help control model complexity and prevent the model from fitting noise in the training data.

The tuned model achieved the following performance:

- Train R²: 0.9205  
- Test R²: 0.8925  
- Train MAPE: 12.75%  
- Test MAPE: 14.58%  
- Train MdAPE: 9.21%  
- Test MdAPE: 10.12%  

The test R² value of **0.8925** indicates that the model explains nearly **89% of the variance in the test dataset**, suggesting strong predictive performance. The relatively small gap between training and testing R² suggests that the model generalizes well and is not severely overfitting.

The error metrics also provide insight into prediction accuracy. A **test MAPE of 14.58%** means that predictions differ from actual values by about 14.6% on average, while the **median absolute percentage error (MdAPE) of 10.12%** indicates that most predictions have even smaller errors. The difference between mean and median errors suggests that the majority of predictions are relatively accurate, with a small number of larger errors.

Overall, hyperparameter tuning improved the LightGBM model by identifying a parameter configuration that achieves a strong balance between predictive accuracy and generalization on unseen data.


# END : LEXI CHEN